# SAHAI: three-arm paired benchmark

Answers the one question every earlier evaluation left open: **is the trained
tutor better than the model it started from?**

| arm | what it is |
|---|---|
| `unaided` | student alone, empty dialogue |
| `base` | the same checkpoint with LoRA **disabled** |
| `trained` | the trained adapter |

The `base` arm is what makes this an ablation rather than a demo. "Trained
tutor beats no tutor" cannot separate the value of the training from the value
of having any tutor in the loop; with the adapters switched off the weights are
identical, so arms 2 and 3 differ by the training and nothing else.

All three arms run on the **same problems in the same order with the same
seed**, and are compared with McNemar's exact test on the problems where two
arms disagree. Problem difficulty dominates this metric -- a 20-problem mean
moved +-0.08 purely on which problems were drawn, larger than the entire
6.2%-16.3% spread across six runs -- so pairing removes that variance instead
of averaging over it.

**Read the power line the benchmark prints before reading its result.** At a
true rate near 0.15, 60 problems cannot resolve an improvement smaller than
about 22 points. A null result here means "not resolvable at this n", not "no
difference".

## 1. Environment

In [ ]:
# Kaggle preinstalls torch/transformers/peft/bitsandbytes/datasets. Uncomment
# only if an import below actually fails.
# !pip install -q transformers peft accelerate bitsandbytes pydantic typer datasets

# Kaggle's image ships torchao 0.10.0, but the installed peft requires >= 0.16.0
# and its is_torchao_available() *raises* ImportError instead of returning False.
# That kills get_peft_model() even though this project never uses torchao
# quantization. Removing the package makes peft skip the torchao dispatcher
# cleanly and fall through to the standard LoRA path.
import importlib.metadata
import importlib.util
import subprocess
import sys


def drop_incompatible_torchao(minimum=(0, 16, 0)):
    if importlib.util.find_spec("torchao") is None:
        print("torchao not installed — nothing to do.")
        return
    try:
        raw = importlib.metadata.version("torchao")
        parsed = tuple(int(p) for p in raw.split(".")[:3])
    except Exception:
        raw, parsed = "unknown", (0, 0, 0)

    if parsed >= minimum:
        print(f"torchao {raw} is compatible with peft — leaving it installed.")
        return

    print(f"Removing torchao {raw} (peft needs >= {'.'.join(map(str, minimum))})...")
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
        check=False,
    )
    importlib.invalidate_caches()
    still_there = importlib.util.find_spec("torchao") is not None
    print("torchao removed." if not still_there else "WARNING: torchao still importable.")


drop_incompatible_torchao()

def ensure_bitsandbytes(minimum="0.46.1"):
    """`Settings.kaggle()` sets `student_quantize_4bit=True`, which needs this.

    Historically preinstalled on Kaggle's image; stopped being true at some
    point and broke a run at the very first model load with a plain
    ImportError. Checking and installing defensively means a future image
    change fails the same way silently instead of wasting a GPU session.
    """
    if importlib.util.find_spec("bitsandbytes") is not None:
        print("bitsandbytes already installed — nothing to do.")
        return
    print(f"bitsandbytes missing — installing >= {minimum}...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"bitsandbytes>={minimum}"],
        check=True,
    )
    importlib.invalidate_caches()
    ok = importlib.util.find_spec("bitsandbytes") is not None
    print("bitsandbytes installed." if ok else "WARNING: bitsandbytes still missing.")


ensure_bitsandbytes()


def ensure_datasets(minimum="2.14.0"):
    """`load_mbpp` needs this; docs already flagged it as unreliably present
    alongside bitsandbytes, so check it the same defensive way."""
    if importlib.util.find_spec("datasets") is not None:
        print("datasets already installed — nothing to do.")
        return
    print(f"datasets missing — installing >= {minimum}...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"datasets>={minimum}"],
        check=True,
    )
    importlib.invalidate_caches()
    ok = importlib.util.find_spec("datasets") is not None
    print("datasets installed." if ok else "WARNING: datasets still missing.")


ensure_datasets()


## 2. Locate the sahai package

In [ ]:
import os
import sys
import zipfile


def find_sahai_root(base="/kaggle/input"):
    """Locate the directory containing the sahai package.

    The mount path is not predictable: a dataset attached via
    kernel-metadata.json lands under /kaggle/input/datasets/<owner>/<slug>/,
    while one attached in the browser lands at /kaggle/input/<slug>/. Walk
    instead of guessing, and fall back to extracting sahai.zip if Kaggle
    served the archive without expanding it.
    """
    if not os.path.isdir(base):
        return None  # running locally

    for root, dirs, files in os.walk(base):
        if os.path.basename(root) == "sahai" and "__init__.py" in files:
            return os.path.dirname(root)

    for root, dirs, files in os.walk(base):
        if "sahai.zip" in files:
            dest = "/kaggle/working/_sahai_src"
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(os.path.join(root, "sahai.zip")) as zf:
                zf.extractall(dest)
            print(f"Extracted sahai.zip -> {dest}")
            # The archive may or may not contain a top-level sahai/ directory.
            if os.path.isdir(os.path.join(dest, "sahai")):
                return dest
            return os.path.dirname(dest)

    return None


sahai_root = find_sahai_root()
if sahai_root:
    sys.path.insert(0, sahai_root)
    print(f"Found sahai package in: {sahai_root}")
elif os.path.isdir("/kaggle/input"):
    print("ERROR: sahai package not found. Full /kaggle/input tree:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count(os.sep) - 2
        print(f"{'  ' * depth}{os.path.basename(root) or root}/")
        for f in sorted(files)[:10]:
            print(f"{'  ' * (depth + 1)}{f}")

import sahai
print(f"SAHAI version: {sahai.__version__}")

## 3. GPU check

A P100 cannot run this stack: Kaggle's PyTorch build has no sm_60 kernels. Fail fast with a clear message rather than deep inside model loading.

In [ ]:
import torch
from sahai.core.models import load_for_training, load_for_inference

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    capability = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"Compute capability: sm_{capability[0]}{capability[1]}")

    # Kaggle's PyTorch build has no sm_60 kernels, and bitsandbytes 4-bit needs
    # Turing or newer, so a P100 cannot run this stack at all. Stop immediately
    # rather than failing later inside model loading with an opaque
    # "no kernel image is available for execution on the device".
    #
    # Note: `--accelerator gpu-t4x2` is accepted by the CLI but ignored by the
    # server, so the accelerator must be set in the Kaggle notebook sidebar.
    if capability < (7, 0):
        raise RuntimeError(
            f"This GPU (sm_{capability[0]}{capability[1]}) cannot run this stack: "
            f"PyTorch here requires sm_70+ and bitsandbytes 4-bit requires sm_75+. "
            f"Open the notebook on kaggle.com and set Accelerator = 'GPU T4 x2' "
            f"in the right-hand sidebar, then Save & Run All."
        )
else:
    raise RuntimeError("No GPU. Set enable_gpu in kernel-metadata.json.")

## 4. Locate the trained adapter

The adapter comes from the training kernel's output, attached as a data source
in `kernel-metadata.json`. Walk for it rather than hardcoding a path: the mount
point differs depending on how the source was attached.

In [ ]:
import os


def find_adapter(base="/kaggle/input"):
    """Find a saved PEFT adapter directory under the input mounts.

    Prefers `best_model` over `final_model`. That ordering is the whole point
    of the early-stopping work: `final_model` is simply the last epoch, and in
    the v24 run epoch 5 scored better on training rollouts than the epoch 9
    that got shipped. If a run produced a deliberately selected checkpoint,
    benchmark that one.
    """
    hits = []
    for root, _dirs, files in os.walk(base):
        if "adapter_config.json" in files:
            hits.append(root)
    if not hits:
        return None
    for preferred in ("best_model", "final_model"):
        for h in hits:
            if os.path.basename(h) == preferred:
                return h
    return sorted(hits)[0]


ADAPTER = find_adapter()
print("adapter:", ADAPTER)
if ADAPTER:
    import json as _json
    cfg = _json.load(open(os.path.join(ADAPTER, "adapter_config.json")))
    print("  lora_dropout:", cfg.get("lora_dropout"),
          " r:", cfg.get("r"), " alpha:", cfg.get("lora_alpha"))
    print("  base model  :", cfg.get("base_model_name_or_path"))
    sel = os.path.join(ADAPTER, "selection.json")
    if os.path.exists(sel):
        print("  selected by early stopping:", open(sel).read().strip())
    else:
        print("  NOTE: no selection.json -- this is a last-epoch checkpoint,")
        print("        not one chosen on held-out score.")
else:
    raise SystemExit(
        "No adapter found under /kaggle/input. Attach the training kernel "
        "(aswathiranjith/sahai-grpo-training) as a data source, or a dataset "
        "containing final_model/."
    )

## 5. Run both framings

Two passes on the same 60 problems. `full` replays the whole transcript, with
the student's own turns as its assistant history. `hints` passes only the
tutor's turns as a single user message.

Both run under the same fixed solve prompt, so the comparison isolates the
framing. Running `hints` alone would confound it with the prompt fixes landed
alongside it, and three variables moving at once is how v14 produced a result
nothing could be attributed to.

Budget: ~8.5 min unaided plus four tutored arms at ~54 min, about 3.7 h against
the 12 h cap.

In [ ]:
import importlib, logging, os, sys, time

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)


def run_pass(mode, arms, out):
    """One benchmark pass under a given solve-context framing.

    Re-imports the module so SAHAI_SOLVE_CONTEXT is read fresh; the student
    checks it per call, but the module-level import is cached and re-importing
    costs nothing next to 40 minutes of generation.
    """
    os.environ["SAHAI_SOLVE_CONTEXT"] = mode
    sys.argv = ["ab_benchmark", "--problems", "60", "--split", "test",
                "--adapter", ADAPTER, "--arms", arms,
                "--solve-context", mode, "--out", out]
    import sahai.eval.ab_benchmark as ab
    importlib.reload(ab)
    t = time.time()
    ab.main()
    print(f"\n=== pass '{mode}' finished in {(time.time()-t)/60:.1f} min ===\n")


t0 = time.time()

# Pass 1 carries the unaided arm. Under `hints` an empty dialogue contributes
# no tutor turns, so unaided would be byte-identical and running it twice would
# buy nothing but 8 minutes.
run_pass("full",  "unaided,base,trained", "/kaggle/working/ab_full.json")
run_pass("hints", "base,trained",         "/kaggle/working/ab_hints.json")

print(f"TOTAL {(time.time()-t0)/60:.1f} min")

## 6. Result


In [ ]:
import json

full = json.load(open("/kaggle/working/ab_full.json"))
hints = json.load(open("/kaggle/working/ab_hints.json"))


def rate(d, arm):
    s = d["arms"][arm]["solved"]
    return sum(s) / max(len(s), 1)


def partial(d, arm):
    p = d["arms"][arm]["partial"]
    return sum(p) / max(len(p), 1)


base_line = rate(full, "unaided")
print(f"unaided (no dialogue at all)      solve {base_line:.3f}  partial {partial(full,'unaided'):.3f}")
print()
print(f"{'arm':<10} {'full':>18} {'hints':>18}")
print("-" * 48)
for arm in ("base", "trained"):
    print(f"{arm:<10} {rate(full,arm):>10.3f} solve {rate(hints,arm):>12.3f} solve")
    print(f"{'':<10} {partial(full,arm):>10.3f} part  {partial(hints,arm):>12.3f} part")

print("\n=== does the hints framing recover the gap? ===")
from sahai.eval.ab_benchmark import mcnemar

for arm in ("base", "trained"):
    m = mcnemar(full["arms"][arm]["solved"], hints["arms"][arm]["solved"])
    d = rate(hints, arm) - rate(full, arm)
    print(f"  {arm:<8} full -> hints  delta={d:+.3f}  "
          f"hints won {m['b01']}, full won {m['b10']}, p={m['p_value']:.4f}")

print("\n=== and against no tutor at all? ===")
for tag, d in (("full", full), ("hints", hints)):
    for arm in ("base", "trained"):
        m = mcnemar(full["arms"]["unaided"]["solved"], d["arms"][arm]["solved"])
        gap = rate(d, arm) - base_line
        verdict = "SIGNIFICANT" if m["p_value"] < 0.05 else "not resolvable"
        print(f"  unaided -> {arm}({tag}):  delta={gap:+.3f}  p={m['p_value']:.4f}  [{verdict}]")

print("\nIf the gap to unaided closes under `hints`, the harm was the framing:")
print("the student was being conditioned on its own expressed confusion.")
print("If it does not close, the framing is not the cause and the simulated")
print("student itself is the thing to re-examine.")